## AGU figure guidelines

From the AGU journal graphic requirements, https://www.agu.org/Publish-with-AGU/Publish/Author-Resources/Graphic-Requirements

- Size
    - One column: 50 to 85 mm wide (1.97 to 3.35 in)
    - Two columns: 105 to 170 mm wide (4.13 to 6.69 in)
    - Height: 228 mm at most (8.98 in)
- Text
    - Font: Arial, Helvetica, Times, or Symbol, or the text is outlined or embedded
    - Size: close to 8 pt at print size, 6 pt for subscripts and superscripts
- Lines: 0.5 pt or thicker, no hairlines
- Raster resolution: 300 to 600 ppi at print size
- Color
    - RGB mode
    - Readable with a color vision deficiency: no rainbow and no red-green palettes
    - Do not rely on color alone: pair it with line styles, symbols, or patterns
- File format: JPG, TIFF, EPS, PS, or PDF
- Submission
    - First submission: figures inline with the text and caption
    - Revision: each main-text figure as its own file, supplement figures inline with their captions

In [ ]:
from pathlib import Path
from typing import cast
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.utils.analysis_period import restrict_to_analysis_period
from eddy_tracking.config import load_analysis_config, read_settings, resolve_bronze_dir, resolve_gold_dir, resolve_output_dir
from eddy_tracking.preprocess.streamline import trace_mean_streamline

EXPERIMENT = 'gulf_stream_20240305_20260531'
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_analysis_config(EXPERIMENT)
eddy_id_settings = read_settings(resolve_output_dir(EXPERIMENT, 'eddy_id'), 'identify-eddies')
axis_settings = read_settings(resolve_output_dir(EXPERIMENT, 'gulf_stream'), 'trace-axis')

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    # matplotlib 3.11 warns about its default calligraphic face, Apple Chancery, even though no label uses it
    'mathtext.cal': 'Arial',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

In [ ]:
from datetime import date

from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.figure import Figure, SubFigure
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import MaxNLocator
from eddy_tracking.preprocess.swot import index_swot_files_by_date
from eddy_tracking.preprocess.streamline import compute_signed_distance_km
from eddy_tracking.preprocess.tracks import load_track_observations
from eddy_tracking.utils.analysis_period import read_analysis_period

REGION_DATE = date(2025, 1, 24)
CONTOUR_EVERY_DAYS = 10
lon_range = eddy_id_settings['lon']
lat_range = eddy_id_settings['lat']
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
age_cmaps = {'cyclone': ListedColormap(plt.get_cmap('Blues')(np.linspace(0.3, 1, 256))), 'anticyclone': ListedColormap(plt.get_cmap('Reds')(np.linspace(0.3, 1, 256)))}  # pyright: ignore[reportAttributeAccessIssue]

swot_files = sorted(resolve_bronze_dir(EXPERIMENT, 'swot_l4_open_ocean').glob('*.nc'))
swot_by_date = index_swot_files_by_date(resolve_bronze_dir(EXPERIMENT, 'swot_l4_open_ocean'))
with xr.open_dataset(swot_by_date[REGION_DATE]) as ds:
    region_adt = ds['adt_full'].to_numpy()[0]
    lon = ds['longitude'].to_numpy()
    lat = ds['latitude'].to_numpy()

# Trace the GS axis on the time-mean ADT field, at the level with the fastest mean flow along it
streamline = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet')
mean_axis = trace_mean_streamline(swot_files, tuple(axis_settings['adt_levels']))
axis_lon = mean_axis.lon
axis_lat = mean_axis.lat

eddy_tracks = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet'), cfg)
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['is_target'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['lifetime_days'] = (eddy_tracks['death_date'] - eddy_tracks['birth_date']).dt.days
track_observations = load_track_observations(resolve_output_dir(EXPERIMENT, 'eddy_track'))
# Panel (c) shows the case study eddies picked on 2026-10-14 (docs/case_study_eddies.md). Each is found by polarity and its
# center on the day of maximum amplitude, not by track id, so the picks survive a re-track that renumbers the tracks.
CASE_STUDY_PICKS = (
    ('cyclone', '2025-12-24', -65.12, 36.53),
    ('anticyclone', '2024-06-29', -63.56, 39.92),
)
MAX_PICK_OFFSET_DEG = 0.5


def find_track_id(polarity: str, max_day: str, lon: float, lat: float) -> int:
    day = track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['date'].eq(pd.Timestamp(max_day))]
    # Distance in degrees of latitude, with longitude scaled by cos(latitude)
    offset = np.hypot((day['center_lon'].to_numpy() - lon) * np.cos(np.radians(lat)), day['center_lat'].to_numpy() - lat)
    if offset.size == 0 or offset.min() > MAX_PICK_OFFSET_DEG:
        raise ValueError(f'no {polarity} within {MAX_PICK_OFFSET_DEG} degrees of the pick on {max_day}')
    track_id = int(day['track_id'].to_numpy()[offset.argmin()])
    if not eddy_tracks.loc[eddy_tracks['polarity'].eq(polarity) & eddy_tracks['track_id'].eq(track_id), 'is_target'].any():
        raise ValueError(f'{polarity} {track_id} is not a target eddy')
    return track_id


example_ids = {polarity: find_track_id(polarity, max_day, lon, lat) for polarity, max_day, lon, lat in CASE_STUDY_PICKS}
example_tracks = {
    polarity: cast(pd.DataFrame, track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(example_ids[polarity])]).sort_values('date')
    for polarity in polarity_colors
}
# The target class is judged against each day's axis, so panel (c) draws the axis of the day each case eddy first reaches its
# destination side; against the time-mean axis a crossing can be invisible
destination_signs = {'cyclone': -1, 'anticyclone': 1}  # signed distance is positive north of the axis
daily_axes = {date: day.sort_values('point_idx') for date, day in streamline.groupby('date')}
crossing_axes = {}
for polarity, track in example_tracks.items():
    signed_km = np.array([
        compute_signed_distance_km(daily_axes[date]['lon'].to_numpy(), daily_axes[date]['lat'].to_numpy(), x, y)[0]
        for date, x, y in zip(track['date'], track['center_lon'], track['center_lat'])
    ])
    on_destination = destination_signs[polarity] * signed_km > 0
    crossing_date = cast(pd.Timestamp, track['date'].iloc[int(np.flatnonzero(on_destination[1:] & ~on_destination[:-1])[0]) + 1])
    crossing_axes[polarity] = (crossing_date, daily_axes[crossing_date])
# The two case eddies live at different times, so panel (b) shows the day, within either one's analysis-period life, with the most target eddies
target_keys = eddy_tracks.loc[eddy_tracks['is_target'], ['polarity', 'track_id']]
target_days = track_observations.merge(target_keys, on=['polarity', 'track_id'])
analysis_start, analysis_end = read_analysis_period(cfg)
case_days = pd.DatetimeIndex(np.concatenate([track['date'].to_numpy() for track in example_tracks.values()])).unique()
case_days = case_days[(case_days >= analysis_start) & (case_days <= analysis_end)]
snapshot_date = cast(pd.Timestamp, target_days.loc[target_days['date'].isin(case_days)].groupby('date').size().idxmax())
with xr.open_dataset(swot_by_date[snapshot_date.date()]) as ds:
    sla = ds['sla'].to_numpy()[0]
    sla_lon = ds['longitude'].to_numpy()
    sla_lat = ds['latitude'].to_numpy()

all_lon = np.concatenate([np.concatenate(track['contour_lon'].to_list()) for track in example_tracks.values()])
all_lat = np.concatenate([np.concatenate(track['contour_lat'].to_list()) for track in example_tracks.values()])
# Panel (c) keeps the aspect ratio of panel (b), so the two maps are the same height side by side; the box covers every contour
# with a margin and grows along its short side to reach that ratio, so no contour is clipped
map_aspect = (lat_range[1] - lat_range[0]) / (lon_range[1] - lon_range[0])
CASE_MARGIN_DEG = 1.0
example_lon = [all_lon.min() - CASE_MARGIN_DEG, all_lon.max() + CASE_MARGIN_DEG]
example_lat = [all_lat.min() - CASE_MARGIN_DEG, all_lat.max() + CASE_MARGIN_DEG]
lon_span, lat_span = example_lon[1] - example_lon[0], example_lat[1] - example_lat[0]
if lat_span < lon_span * map_aspect:
    grow = (lon_span * map_aspect - lat_span) / 2
    example_lat = [example_lat[0] - grow, example_lat[1] + grow]
else:
    grow = (lat_span / map_aspect - lon_span) / 2
    example_lon = [example_lon[0] - grow, example_lon[1] + grow]
example_extent = (*example_lon, *example_lat)

fig = plt.figure(figsize=(6.69, 6.8))
region_fig, tracks_fig = cast(list[SubFigure], fig.subfigures(2, 1, height_ratios=(3.75, 3.05)))
ax = cast(GeoAxes, region_fig.add_axes((0.065, 0.06, 0.81, 0.88), projection=ccrs.PlateCarree()))  # pyright: ignore[reportArgumentType]
ax.set_extent([*lon_range, *lat_range], crs=ccrs.PlateCarree())
adt_fill = ax.contourf(lon, lat, region_adt, levels=np.arange(-1.25, 1.2501, 0.05), cmap='RdBu_r', transform=ccrs.PlateCarree(), zorder=1)
ax.add_feature(cfeature.LAND.with_scale('10m'), facecolor='#e8e8e8', zorder=2)
ax.coastlines(resolution='10m', color='#4d4d4d', linewidth=0.5, zorder=3)
for name, name_lon, name_lat in (('Gulf Stream', -70, 38.5), ('Sargasso Sea', -62, 30.5)):
    ax.text(name_lon, name_lat, name, fontsize=9, fontstyle='italic', color='#262626', ha='center', va='center', transform=ccrs.PlateCarree(), zorder=5)
gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='#808080', alpha=0.5, xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2), zorder=4)
gridlines.top_labels = False
gridlines.right_labels = False
gridlines.xlabel_style = {'size': 8}
gridlines.ylabel_style = {'size': 8}
adt_bar = fig.colorbar(adt_fill, cax=ax.inset_axes((1.02, 0.125, 0.025, 0.75)))
adt_bar.set_ticks([-1, -0.5, 0, 0.5, 1], labels=['−1', '−0.5', '0', '0.5', '1'])
adt_bar.set_label('Absolute dynamic topography (m)')
adt_bar.outline.set_linewidth(0.6)  # pyright: ignore[reportAttributeAccessIssue]
inset = cast(GeoAxes, ax.inset_axes((0.03, 0.68, 0.24, 0.3), projection=ccrs.PlateCarree()))
inset.set_extent([-100, -30, 10, 55], crs=ccrs.PlateCarree())
inset.add_feature(cfeature.LAND.with_scale('110m'), facecolor='#e8e8e8', edgecolor='#808080', linewidth=0.3)
inset.add_feature(cfeature.OCEAN.with_scale('110m'), facecolor='#dbe9f6')
inset.add_patch(Rectangle((lon_range[0], lat_range[0]), lon_range[1] - lon_range[0], lat_range[1] - lat_range[0], linewidth=1, edgecolor='red', facecolor='red', alpha=0.25, transform=ccrs.PlateCarree()))
ax.set_title('$\\bf{(a)}$', loc='left')

maps = tracks_fig.add_gridspec(1, 2, left=0.065, right=0.99, bottom=0.25, top=0.93, wspace=0.19)
snapshot_ax = cast(GeoAxes, tracks_fig.add_subplot(maps[0], projection=ccrs.PlateCarree()))
snapshot_ax.set_extent([*lon_range, *lat_range], crs=ccrs.PlateCarree())
sla_mesh = snapshot_ax.pcolormesh(sla_lon, sla_lat, sla, cmap='RdBu_r', vmin=-0.8, vmax=0.8, alpha=0.8, shading='nearest', rasterized=True, transform=ccrs.PlateCarree(), zorder=1)
sla_bar = fig.colorbar(sla_mesh, cax=snapshot_ax.inset_axes((0, -0.2, 1, 0.045)), orientation='horizontal')
sla_bar.set_label('Sea level anomaly (m)')
sla_bar.set_ticks([-0.8, -0.4, 0, 0.4, 0.8], labels=['−0.8', '−0.4', '0', '0.4', '0.8'])
sla_bar.ax.tick_params(length=2)
sla_bar.outline.set_linewidth(0.6)  # pyright: ignore[reportAttributeAccessIssue]
axis = streamline.loc[streamline['date'].eq(snapshot_date)].sort_values('point_idx')
snapshot_ax.plot(axis['lon'], axis['lat'], color='black', linewidth=1.0, transform=ccrs.PlateCarree(), zorder=4)
today = track_observations.loc[track_observations['date'].eq(snapshot_date)].merge(eddy_tracks[['polarity', 'track_id', 'is_target']], on=['polarity', 'track_id'])
for polarity, contour_lon, contour_lat, is_target in zip(today['polarity'], today['contour_lon'], today['contour_lat'], today['is_target']):
    snapshot_ax.plot(np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]), color=polarity_colors[polarity], linewidth=1.4 if is_target else 0.7, linestyle='solid' if is_target else (0, (3, 1.5)), transform=ccrs.PlateCarree(), zorder=5)
snapshot_ax.set_title('$\\bf{(b)}$', loc='left')
snapshot_ax.legend(handles=[
    Line2D([], [], color=polarity_colors['cyclone'], linewidth=1.4, label='Cyclone'),
    Line2D([], [], color=polarity_colors['anticyclone'], linewidth=1.4, label='Anticyclone'),
    Line2D([], [], color='#444444', linewidth=1.4, label='Target eddy'),
    Line2D([], [], color='#444444', linewidth=0.7, linestyle=(0, (3, 1.5)), label='Nontarget eddy'),
    Line2D([], [], color='black', linewidth=1.0, label='Gulf Stream axis'),
], loc='lower right', frameon=True, framealpha=0.9, edgecolor='none', fontsize=6.5, handlelength=1.8, borderpad=0.3, labelspacing=0.15)

# Panel (c): both example tracks on one map, each contour every CONTOUR_EVERY_DAYS days, colored by age
example_ax = cast(GeoAxes, tracks_fig.add_subplot(maps[1], projection=ccrs.PlateCarree()))
example_ax.set_extent(example_extent, crs=ccrs.PlateCarree())
for polarity, (crossing_date, crossing_axis) in crossing_axes.items():
    example_ax.plot(crossing_axis['lon'], crossing_axis['lat'], color=polarity_colors[polarity], linewidth=1.1, linestyle='--', transform=ccrs.PlateCarree(), zorder=6)
for panel, (polarity, track) in enumerate(example_tracks.items()):
    first_date, last_date = track['date'].iloc[0], track['date'].iloc[-1]
    age_days = (track['date'] - first_date).dt.days
    age_norm = Normalize(0, int(age_days.iloc[-1]))
    drawn = track.loc[age_days.mod(CONTOUR_EVERY_DAYS).eq(0) | track['date'].eq(last_date)]
    for contour_lon, contour_lat, age in zip(drawn['contour_lon'], drawn['contour_lat'], age_days.loc[drawn.index]):
        example_ax.plot(np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]), color=age_cmaps[polarity](age_norm(age)), linewidth=0.7, transform=ccrs.PlateCarree(), zorder=5)
    age_bar = fig.colorbar(ScalarMappable(norm=age_norm, cmap=age_cmaps[polarity]), cax=example_ax.inset_axes((0.53 * panel, -0.2, 0.47, 0.045)), orientation='horizontal')
    age_bar.set_label(f'{polarity.capitalize()} age (days)')
    age_bar.ax.xaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 5, 10]))
    age_bar.ax.tick_params(length=2)
    age_bar.outline.set_linewidth(0.6)  # pyright: ignore[reportAttributeAccessIssue]
    here = today.loc[today['polarity'].eq(polarity) & today['track_id'].eq(example_ids[polarity])]
    for contour_lon, contour_lat in zip(here['contour_lon'], here['contour_lat']):
        snapshot_ax.text(contour_lon.max() + 0.2, contour_lat.max(), 'c', fontweight='bold', transform=ccrs.PlateCarree(), zorder=6)
example_ax.set_title('$\\bf{(c)}$', loc='left')
example_ax.legend(handles=[
    Line2D([], [], color=polarity_colors[polarity], linewidth=1.1, linestyle='--', label=f'Axis, {crossing_date.day} {crossing_date:%b %Y}')
    for polarity, (crossing_date, _) in crossing_axes.items()
], loc='lower left', frameon=True, framealpha=0.9, edgecolor='none', fontsize=6.5, handlelength=1.8, borderpad=0.3, labelspacing=0.15)
for ax in (snapshot_ax, example_ax):
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=2)
    ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
    gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2), zorder=3)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.xlabel_style = {'size': 8}
    gridlines.ylabel_style = {'size': 8}
print(eddy_tracks.loc[eddy_tracks['is_target']].groupby('polarity').size().rename('target eddies').to_string())
print(f"case study eddies: cyclone {example_ids['cyclone']}, anticyclone {example_ids['anticyclone']}; panel (b) date {snapshot_date:%Y-%m-%d}; crossing days {', '.join(f'{k} {v[0]:%Y-%m-%d}' for k, v in crossing_axes.items())}")
plt.show()

Figure 1. (a) Absolute dynamic topography across the study region (81°W to 56°W, 29°N to 44°N). The analysis period, 5 March 2024 to 1 June 2026, is the time covered by the SSH, PACE, and Copernicus records together. (b) Eddy snapshot: target eddies are outlined in bold while nontarget eddies have dashed outlines. The analysis period contains 10 target cyclones and 20 target anticyclones (printed by the cell above). (c) Speed contours of the two case study eddies, target cyclone and target anticyclone, drawn once every 10 days and colored by days since first detection. Each dashed line is the Gulf Stream axis on the day that eddy first reached the far side of it, in the eddy's color. The eddies are found by their position on the day of maximum amplitude (docs/case_study_eddies.md), so a re-track that renumbers the tracks does not change them. Panel (b) shows the day, within either case eddy's life in the analysis period, with the most target eddies, and marks a case eddy alive that day with c.

Goal (a): Display the region that we're focusing on

Goal (b, c): Display the eddy tracking and contour identification, which types of eddies the study targets, and an example of eddy track progression over an eddy's lifetime.

In [ ]:
from matplotlib.ticker import MaxNLocator
from eddy_tracking.utils.plankton_layout import COPERNICUS_LABELS, SIZE_CLASS_CELLS, SIZE_CLASS_FIELDS, SIZE_CLASS_GRID_SHAPE, find_lowest_rows, find_spare_cells
from eddy_tracking.utils.phase_position import add_phase_position, measure_distance_past_axis
from eddy_tracking.validation.quality import mask_invalid_plankton_groups, keep_valid_sdp_rows

N_AGE_BINS = 5
# Bins on each side of the two-part axis, two because only 10 to 18 target eddies fill them, and how far past the axis an eddy must get before its time on the new side starts
N_PHASE_BINS = 2
NEAR_AXIS_KM = 150
N_RADIAL_BINS = 5
MAX_RADIUS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
identity_columns = ['polarity', 'track_id']
shown_pigments = {
    'Tchla': 'Total chlorophyll-a', 'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
    'ButFuco': "19'-But-fucoxanthin", 'HexFuco': "19'-Hex-fucoxanthin", 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin', 'Fuco': 'Fucoxanthin',
    'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Perid': 'Peridinin',
}
# Size classes over the groups they sum, in the column order of plankton_layout; NANO is left out because it equals HAPTO on every row of the plankton gold table
copernicus_pfts = {field: COPERNICUS_LABELS[field] for field in SIZE_CLASS_FIELDS}
group_styles = {
    'target': {'linestyle': '-', 'linewidth': 1.2, 'marker': 'o', 'markeredgecolor': 'white', 'markeredgewidth': 0.5},
    'nontarget': {'linestyle': (0, (3, 1.5)), 'linewidth': 0.9, 'marker': 's', 'markerfacecolor': 'white', 'markeredgewidth': 0.7, 'alpha': 0.75},
}
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)
phase_edges = np.linspace(0, 2, 2 * N_PHASE_BINS + 1)
phase_centers = (phase_edges[:-1] + phase_edges[1:]) / 2
phase_ticks = ([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1 | 0', '0.5', '1'])
polarity_sides = {'cyclone': ('north', 'south'), 'anticyclone': ('south', 'north')}

eddy_groups = eddy_tracks[identity_columns].assign(group=np.where(eddy_tracks['is_target'], 'target', 'nontarget'))
plankton = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet'), cfg).merge(eddy_groups, on=identity_columns)
# The gold table gates on CHL coverage, and the PFT fields have a smaller valid mask, so each PFT gets the same gate on its own pixels
plankton_settings = read_settings(resolve_gold_dir(EXPERIMENT, 'eddy_plankton_table.parquet'), 'collocate-plankton')
plankton = mask_invalid_plankton_groups(plankton, plankton_settings)
pigments = keep_valid_sdp_rows(restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet'), cfg))
pigments['polarity'] = cast(pd.Series, pigments['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
pigments = pigments.merge(eddy_groups.loc[eddy_groups['group'].eq('target')], on=identity_columns)
plankton_rings = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_rings.parquet'), cfg).dropna(subset=['CHL'])
plankton_rings = cast(pd.DataFrame, plankton_rings.assign(radial_bin=plankton_rings['radial_bin'] // 2, CHL=plankton_rings['CHL'] * plankton_rings['CHL_n_pixels']).groupby(identity_columns + ['date', 'radial_bin'])[['CHL', 'CHL_n_pixels']].sum()).reset_index()
plankton_rings['CHL'] /= plankton_rings['CHL_n_pixels']

analysis = pd.concat([
    plankton[identity_columns + ['group', 'date', 'age_frac']].assign(source='chl', variable='CHL', concentration=plankton['CHL']),
    # The PFTs come from the same Copernicus composites as CHL, so they share its source and its bootstrap draws
    plankton[identity_columns + ['group', 'date', 'age_frac'] + list(copernicus_pfts)].melt(id_vars=identity_columns + ['group', 'date', 'age_frac'], var_name='variable', value_name='concentration').dropna(subset=['concentration']).assign(source='chl'),
    pigments[identity_columns + ['group', 'date', 'age_frac']].assign(source='sdp').join(
        cast(pd.DataFrame, pigments[[f'eddy_mean_{pigment}' for pigment in shown_pigments]]).rename(columns={f'eddy_mean_{pigment}': pigment for pigment in shown_pigments}),
    ).join(
        # Each ratio divides one interior mean by the other, the unit of the across eddy-composites tree in pigments_clustering.ipynb; a mean of pixel ratios blows up where Tchla is near zero
        cast(pd.DataFrame, pigments[[f'eddy_mean_{pigment}' for pigment in shown_pigments]]).div(pigments['eddy_mean_Tchla'], axis=0).rename(columns={f'eddy_mean_{pigment}': f'{pigment}_ratio' for pigment in shown_pigments}),
    ).melt(id_vars=identity_columns + ['group', 'date', 'age_frac', 'source'], var_name='variable', value_name='concentration'),
], ignore_index=True).sort_values(identity_columns + ['variable', 'date'])
analysis['age_bin'] = np.minimum((analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
analysis['change'] = analysis['concentration'] - analysis.groupby(identity_columns + ['variable'])['concentration'].transform('first')
# Target eddies cross the axis, so they also get a position on the two-part axis: the time on the side they started on, then the time on the side they end on from the first day NEAR_AXIS_KM past the axis, each scaled 0 to 1
crossing_tracks = eddy_tracks.loc[eddy_tracks['is_target']]
crossing_observations = measure_distance_past_axis(resolve_output_dir(EXPERIMENT, 'eddy_track'), resolve_output_dir(EXPERIMENT, 'gulf_stream'), crossing_tracks)
target_phases = add_phase_position(cast(pd.DataFrame, analysis.loc[analysis['group'].eq('target')]), crossing_tracks, crossing_observations, NEAR_AXIS_KM)
binned = pd.concat([
    analysis.assign(time_axis='fraction', position_bin=analysis['age_bin']),
    target_phases.assign(time_axis='phase', position_bin=np.minimum((target_phases['phase_position'] * N_PHASE_BINS).astype(int), 2 * N_PHASE_BINS - 1)),
], ignore_index=True)
bin_positions = {'fraction': bin_centers, 'phase': phase_centers}
# Each variable is summarized both as its change from the eddy's first observation and as its level, which Figure 3 plots for pigment / Tchla
bin_keys = ['time_axis', 'source', 'variable', 'group'] + identity_columns + ['position_bin']
eddy_bins = cast(pd.DataFrame, binned.groupby(bin_keys)[['change', 'concentration']].mean()).reset_index().melt(id_vars=bin_keys, var_name='measure')
n_eddies = binned.groupby(['time_axis', 'source', 'group', 'polarity'])['track_id'].nunique()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for keys, source_bins in eddy_bins.groupby(['time_axis', 'source', 'group', 'polarity']):
    time_axis, source, group, polarity = cast(tuple[str, str, str, str], keys)
    positions = bin_positions[time_axis]
    eddy_ids = sorted(source_bins['track_id'].unique())
    draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
    for variable_keys, variable_bins in source_bins.groupby(['variable', 'measure']):
        variable, measure = cast(tuple[str, str], variable_keys)
        matrix = variable_bins.pivot(index='track_id', columns='position_bin', values='value').reindex(index=eddy_ids, columns=range(len(positions))).to_numpy(dtype=float)
        counts = np.isfinite(matrix).sum(axis=0)
        means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(len(positions), np.nan), where=counts > 0)
        sampled = matrix[draws]
        sampled_counts = np.isfinite(sampled).sum(axis=1)
        sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_counts, out=np.full((N_BOOTSTRAP, len(positions)), np.nan), where=sampled_counts > 0)
        low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
        summary_rows.append(pd.DataFrame({
            'time_axis': time_axis, 'source': source, 'group': group, 'polarity': polarity, 'variable': variable, 'measure': measure, 'position_bin': range(len(positions)),
            'position': positions, 'mean': means, 'ci_low': low, 'ci_high': high, 'n_eddies': counts,
        }))
lifetime_summary = pd.concat(summary_rows, ignore_index=True)

# The heatmaps put target eddies on the two-part axis and nontarget eddies on fraction of observed track
heatmap_rows = (('target', 'phase'), ('nontarget', 'fraction'))
heatmap_composites = binned.loc[binned['variable'].eq('CHL') & pd.MultiIndex.from_frame(cast(pd.DataFrame, binned[['group', 'time_axis']])).isin(heatmap_rows)]
radial = plankton_rings.merge(cast(pd.DataFrame, heatmap_composites[['time_axis'] + identity_columns + ['group', 'date', 'position_bin']]), on=identity_columns + ['date'])
radial_grid = cast(pd.Series, cast(pd.Series, radial.groupby(['time_axis'] + identity_columns + ['group', 'position_bin', 'radial_bin'])['CHL'].mean()).groupby(['group', 'time_axis', 'polarity', 'position_bin', 'radial_bin']).mean())
chl_norm = Normalize(np.floor(radial_grid.min() / 0.02) * 0.02, np.ceil(radial_grid.max() / 0.02) * 0.02)



def draw_change(ax: Axes, source: str, variable: str, series: tuple[tuple[str, str], ...], measure: str = 'change') -> None:
    # series are (group, time_axis) pairs; on a panel with the two-part axis, a group on fraction of observed track spans both parts
    on_phase_axis = any(time_axis == 'phase' for _, time_axis in series)
    if measure == 'change':
        ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    if on_phase_axis:
        ax.axvline(1, color='#999999', linewidth=0.6, zorder=1)
    for polarity, color in polarity_colors.items():
        for group, time_axis in series:
            result = lifetime_summary.loc[lifetime_summary['time_axis'].eq(time_axis) & lifetime_summary['group'].eq(group) & lifetime_summary['polarity'].eq(polarity) & lifetime_summary['variable'].eq(variable) & lifetime_summary['measure'].eq(measure)].sort_values('position_bin')
            x = result['position'] * (2 if on_phase_axis and time_axis == 'fraction' else 1)
            ax.errorbar(x, result['mean'], yerr=[result['mean'] - result['ci_low'], result['ci_high'] - result['mean']], fmt='none', ecolor=color, alpha=1 if group == 'target' else 0.45, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=2)
            ax.plot(x, result['mean'], color=color, markersize=3, label=f'{group.capitalize()} {polarity}s (n = {n_eddies[time_axis, source, group, polarity]})', zorder=4 if group == 'target' else 3, **group_styles[group])
    if on_phase_axis:
        ax.xaxis.set_ticks(*phase_ticks)
        ax.set_xlim(0, 2)
    else:
        ax.xaxis.set_ticks(np.linspace(0, 1, 6).tolist())
        ax.set_xlim(0, 1)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ax.yaxis.set_major_locator(locator)
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    ax.yaxis.set_ticks(ticks.tolist())
    ax.set_ylim(ticks[0], ticks[-1])


def label_sides(ax: Axes) -> None:
    # The first part of the two-part axis is the north side for a cyclone and the south side for an anticyclone, so each part carries one word per polarity in that polarity's color
    for row, (polarity, color) in enumerate(polarity_colors.items()):
        for position, side in zip((0.25, 0.75), polarity_sides[polarity]):
            ax.annotate(f'{side} side', (position, 0), xycoords='axes fraction', xytext=(0, -14 - 9 * row), textcoords='offset points', ha='center', va='top', color=color)


# Layout in inches from the lower left corner: (a) on the left with its legend under it, and the (b) heatmaps on the right, target eddies over nontarget eddies
FIGURE_HEIGHT = 4.15
fig = cast(Figure, plt.figure(figsize=(6.69, FIGURE_HEIGHT)))


def add_axes_in_inches(left: float, bottom: float, width: float, height: float) -> Axes:
    return fig.add_axes((left / 6.69, bottom / FIGURE_HEIGHT, width / 6.69, height / FIGURE_HEIGHT))


change_ax = add_axes_in_inches(0.62, 1.5, 2.58, 1.85)
draw_change(change_ax, 'chl', 'CHL', (('target', 'phase'), ('nontarget', 'fraction')))
change_ax.set_title('$\\bf{(a)}$ Change in CHL', loc='left', pad=34.5)
change_ax.set_ylabel('$\\Delta$CHL (mg m$^{-3}$)')
label_sides(change_ax)
change_ax.set_xlabel('Fraction of time on each side of the Gulf Stream\n(target eddies)', labelpad=21)
# Nontarget eddies never cross, so they keep fraction of observed track, read on the top axis across both parts
top_axis = change_ax.secondary_xaxis('top', functions=(lambda x: np.divide(x, 2), lambda x: np.multiply(x, 2)))
top_axis.xaxis.set_ticks([0, 0.25, 0.5, 0.75, 1], ['0', '0.25', '0.5', '0.75', '1'])
top_axis.tick_params(length=2.5, width=0.6)
top_axis.set_xlabel('Fraction of observed track (nontarget eddies)', labelpad=3)
change_ax.legend(loc='upper left', bbox_to_anchor=(0, -0.47), ncol=2, fontsize=6.8, handlelength=1.5, handletextpad=0.4, borderaxespad=0, labelspacing=0.3, columnspacing=0.5)

heat_cmap = plt.get_cmap('viridis').copy()  # pyright: ignore[reportAttributeAccessIssue]
heat_cmap.set_bad('#e6e6e6')
heat_axes = [[add_axes_in_inches(left, bottom, 1.0, 1.0) for left in (4.45, 5.6)] for bottom in (2.65, 0.95)]
for row, (group, time_axis) in enumerate(heatmap_rows):
    edges = phase_edges if time_axis == 'phase' else bin_edges
    for col, polarity in enumerate(polarity_colors):
        ax = heat_axes[row][col]
        grid = radial_grid.loc[group, time_axis, polarity].unstack('radial_bin').reindex(index=range(len(edges) - 1), columns=range(N_RADIAL_BINS)).to_numpy(dtype=float).T
        ax.pcolormesh(edges, radial_edges, np.ma.masked_invalid(grid), cmap=heat_cmap, norm=chl_norm, edgecolors='white', linewidth=0.5)
        ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
        if time_axis == 'phase':
            ax.xaxis.set_ticks([0, 1, 2], ['0', '1 | 0', '1'])
            for position, side in zip((0.25, 0.75), polarity_sides[polarity]):
                ax.annotate(side, (position, 0), xycoords='axes fraction', xytext=(0, -14), textcoords='offset points', ha='center', va='top', color=polarity_colors[polarity])
        else:
            ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'])
        ax.xaxis.set_ticks(edges.tolist(), minor=True)
        ax.set_xlim(edges[0], edges[-1])
        ax.yaxis.set_ticks([0, 1, 2], ['0', '1', '2'] if col == 0 else [])
        ax.yaxis.set_ticks(radial_edges.tolist(), minor=True)
        ax.tick_params(length=2)
        ax.tick_params(which='minor', length=1.2)
        if row == 0:
            ax.set_title(f'{polarity.capitalize()}s', pad=2)
        if col == 0:
            ax.text(-0.3, 0.5, group.capitalize(), transform=ax.transAxes, rotation=90, ha='center', va='center')
    heat_axes[row][0].set_xlabel('Fraction of time on each side' if time_axis == 'phase' else 'Fraction of observed track', labelpad=13 if time_axis == 'phase' else 3)
    heat_axes[row][0].xaxis.label.set_x(1.075)
heat_axes[0][0].annotate('$\\bf{(b)}$ CHL by radius and time', xy=(0, 1), xycoords='axes fraction', xytext=(0, 13), textcoords='offset points', va='bottom')
fig.text(3.93 / 6.69, 2.3 / FIGURE_HEIGHT, 'Distance from center (speed radii)', rotation=90, ha='center', va='center')
heat_bar = fig.colorbar(ScalarMappable(norm=chl_norm, cmap=heat_cmap), cax=add_axes_in_inches(4.45, 0.38, 2.15, 0.05), orientation='horizontal')
heat_bar.set_label('CHL (mg m$^{-3}$)', labelpad=1)
heat_bar.ax.tick_params(length=2)
heat_bar.ax.xaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]))
heat_bar.outline.set_linewidth(0.5)  # pyright: ignore[reportAttributeAccessIssue]
plt.show()


Figure 2. Change in interior chlorophyll-a (CHL) over the lives of Gulf Stream eddies between 5 March 2024 and 1 June 2026. CHL is the Copernicus multi-sensor product averaged over NASA 8-day periods. The interior value of an eddy is the mean over the pixels inside its speed contour, kept when CHL covers at least half of those pixels and at least 10 of them. Target eddies cross the daily Gulf Stream axis: cyclones from the north side to the south side, and anticyclones from the south side to the north side. All other tracked eddies are nontarget eddies. (a) Change in CHL from each eddy's first observation. Target eddies (solid lines, bottom axis) are placed on a two-part time axis: from 0 to 1 is the time on the side where the eddy was first detected, up to the first day its center lies more than 150 km past the axis, and from 1 to 2 is the remaining time, on the side where it ends. Each part has two bins. Two target anticyclones never reach 150 km past the axis and are left out. Nontarget eddies never cross the axis, so their change (dashed lines, top axis) is binned by the fraction of their observed track, in five bins from first detection (0) to last detection (1). Values are averaged within each eddy and bin, then across eddies, and error bars are 95% bootstrap intervals from resampling eddies. (b) Mean CHL in rings 0.4 speed radii wide around the eddy center, out to two speed radii, for target eddies on the two-part time axis and for nontarget eddies by fraction of observed track, with the bins of (a). Each 0.2-radius ring must hold at least 3 valid pixels and CHL on half of its pixels before pairs of rings are pooled. The dashed line marks one speed radius.

Goal: Show that the interior CHL, pigments, and/or PFT classes change over an eddy's lifetime depending on where it moves

In [ ]:
from eddy_tracking.utils.pigment_clusters import resolve_pigment_clusters

# Okabe and Ito colors, which stay distinct under every color vision deficiency; blue and vermillion are left out because they mark the polarities
PFT_BLOCK_COLORS = ('#E69F00', '#009E73', '#CC79A7', '#56B4E9', '#000000')
N_PIGMENT_COLUMNS = 2
# Inches: the height of one row of panels with its gap, the margins above and below the panel grid, and the left edge of the block labels
PIGMENT_ROW_HEIGHT = 0.99
PIGMENT_MARGIN = 0.525
# Extra room under the panel grid for the side names under the lowest panels
SIDE_LABEL_MARGIN = 0.3
BLOCK_LABEL_X = 0.415
# One block per labeled cluster of the ratio tree in pigments_clustering.ipynb, named after the PFTs whose diagnostic pigments it holds, with the cluster's pigments that occur in at least one of them
resolved_pigments = tuple(
    (cluster.label, PFT_BLOCK_COLORS[block % len(PFT_BLOCK_COLORS)], cluster.carrier_pigments)
    for block, cluster in enumerate(cluster for cluster in resolve_pigment_clusters(DATA_DIR, cfg) if cluster.labels)
)
print('\n'.join(f'{pft_label}: {", ".join(pft_pigments)}' for pft_label, _, pft_pigments in resolved_pigments))
block_rows = [-(-len(pft_pigments) // N_PIGMENT_COLUMNS) for *_, pft_pigments in resolved_pigments]
n_pigment_rows = sum(block_rows)
figure_width = 6.69
figure_height = 2 * PIGMENT_MARGIN + SIDE_LABEL_MARGIN + n_pigment_rows * PIGMENT_ROW_HEIGHT
# Each pigment divided by SDP Tchla, as the level itself
fig = cast(Figure, plt.figure(figsize=(figure_width, figure_height)))
# A block with several PFTs lists one per line; the widest label sets where the brackets and the panels start
block_labels = [fig.text(BLOCK_LABEL_X / figure_width, 0.5, pft_label.replace(', ', ',\n'), color=color, va='center', ha='left') for pft_label, color, _ in resolved_pigments]
fig.draw_without_rendering()
bracket_x = (BLOCK_LABEL_X + max(text.get_window_extent().width for text in block_labels) / fig.dpi + 0.08) / figure_width
pigment_grid = fig.add_gridspec(
    n_pigment_rows, N_PIGMENT_COLUMNS, left=bracket_x + 0.64 / figure_width, right=0.985,
    bottom=(PIGMENT_MARGIN + SIDE_LABEL_MARGIN) / figure_height, top=1 - PIGMENT_MARGIN / figure_height, wspace=0.32, hspace=0.65,
)
pigment_axes = cast(np.ndarray, pigment_grid.subplots(sharex=True, squeeze=False))
bottom_axes: dict[int, Axes] = {}
row = 0
panel = 0
for (pft_label, color, pft_pigments), pft_rows, label_text in zip(resolved_pigments, block_rows, block_labels):
    for offset, pigment in enumerate(pft_pigments):
        ax = cast(Axes, pigment_axes[row + offset // N_PIGMENT_COLUMNS, offset % N_PIGMENT_COLUMNS])
        draw_change(ax, 'sdp', f'{pigment}_ratio', (('target', 'phase'),), 'concentration')
        ax.set_title(f'$\\bf{{({chr(97 + panel)})}}$ {shown_pigments[pigment]} / Tchla', loc='left')
        ax.tick_params(labelbottom=False)
        bottom_axes[offset % N_PIGMENT_COLUMNS] = ax
        panel += 1
    # The cells after the block's last pigment stay empty
    for column in range(len(pft_pigments) - (pft_rows - 1) * N_PIGMENT_COLUMNS, N_PIGMENT_COLUMNS):
        cast(Axes, pigment_axes[row + pft_rows - 1, column]).remove()
    pft_box = pigment_grid[row:row + pft_rows, :].get_position(fig)
    tick = 0.054 / figure_width
    fig.add_artist(Line2D([bracket_x + tick, bracket_x, bracket_x, bracket_x + tick], [pft_box.y0, pft_box.y0, pft_box.y1, pft_box.y1], transform=fig.transFigure, color=color, linewidth=1))  # pyright: ignore[reportAttributeAccessIssue]
    label_text.set_y((pft_box.y0 + pft_box.y1) / 2)
    row += pft_rows
# The lowest panel of each column carries the x tick labels
for ax in bottom_axes.values():
    ax.tick_params(labelbottom=True)
    label_sides(ax)
pigment_box = pigment_grid[:, :].get_position(fig)
fig.legend(*cast(Axes, pigment_axes[0, 0]).get_legend_handles_labels(), loc='upper center', bbox_to_anchor=((pigment_box.x0 + pigment_box.x1) / 2, 1 - 0.035 / figure_height), ncol=2, handlelength=1.8, columnspacing=1)
fig.text(0.054 / figure_width, (pigment_box.y0 + pigment_box.y1) / 2, 'Pigment / Tchla', rotation=90, va='center', ha='left')
fig.text((pigment_box.x0 + pigment_box.x1) / 2, 0.126 / figure_height, 'Fraction of time on each side of the Gulf Stream', va='bottom', ha='center')
plt.show()

Figure 3.

Each block is one cluster of the pigment / Tchla tree over the target eddy-composites (`pigments_clustering.ipynb`, cut at `CLUSTER_CUT_HEIGHT`). The label at left names every PFT whose diagnostic pigment sits in that cluster (Fuco for diatoms, Perid for dinoflagellates, ButFuco for pelagophytes, HexFuco for haptophytes, Allo for cryptophytes, MVchlb and Neo for green algae, Zea and DVchla for cyanobacteria; Kramer and Siegel, 2019), so a block names several PFTs when their markers share one cluster. A block shows the cluster's pigments that occur in at least one of its PFTs. Shared pigments such as Chlc12, Chlc3, and Viola do not name a block, and a cluster with no diagnostic pigment (Chlc3 alone at this cut) is left out. Chrysophytes have no diagnostic pigment among the SDP pigments and cannot be named.

Goal: Show that the interior CHL, pigments, and/or PFT classes change over an eddy's lifetime depending on where it moves

In [ ]:
n_rows, n_columns = SIZE_CLASS_GRID_SHAPE
fig = cast(Figure, plt.figure(figsize=(6.69, 5.6)))
pft_grid = fig.add_gridspec(n_rows, n_columns, left=0.11, right=0.985, bottom=0.135, top=0.97, wspace=0.32, hspace=0.65)
pft_axes = cast(np.ndarray, pft_grid.subplots(sharex=True))
# One column per size class, largest on the left, each over the groups it sums; the lowest panel of each column carries the x tick labels
lowest = find_lowest_rows(SIZE_CLASS_CELLS)
for panel, (row, column, pft) in enumerate(SIZE_CLASS_CELLS):
    ax = cast(Axes, pft_axes[row, column])
    draw_change(ax, 'chl', pft, (('target', 'phase'),))
    ax.set_title(f'$\\bf{{({chr(97 + panel)})}}$ {copernicus_pfts[pft]}', loc='left')
    ax.tick_params(labelbottom=row == lowest[column])
    if row == lowest[column]:
        label_sides(ax)
spare = find_spare_cells(SIZE_CLASS_CELLS, SIZE_CLASS_GRID_SHAPE)
for row, column in spare:
    cast(Axes, pft_axes[row, column]).remove()
legend_box = pft_grid[spare[0]].get_position(fig)
fig.legend(*cast(Axes, pft_axes[0, 0]).get_legend_handles_labels(), loc='center', bbox_to_anchor=((legend_box.x0 + legend_box.x1) / 2, (legend_box.y0 + legend_box.y1) / 2), handlelength=1.8)
pft_box = pft_grid[:, :].get_position(fig)
fig.text(0.008, (pft_box.y0 + pft_box.y1) / 2, 'Change from first observation (mg m$^{-3}$)', rotation=90, va='center', ha='left')
fig.text((pft_box.x0 + pft_box.x1) / 2, 0.022, 'Fraction of time on each side of the Gulf Stream', va='bottom', ha='center')
plt.show()


The Copernicus phytoplankton functional types of the target eddies. Each panel shows the mean change from first observation over two bins of the time on each side of the Gulf Stream, with 95% bootstrap intervals.


Figure 3: Cluster pigments -> Group plots by what PFT they correspond to
Figure 4: Case study on a specific eddy that demonstrates before/after (one when it first crosses into new region, another near end of lifetime in that new region). Plots of all pigments associated with that particular PFT if it works out after clustering results

In [ ]:
from matplotlib.colors import LogNorm
from eddy_tracking.preprocess.streamline import KM_PER_DEG_LAT

N_STRONGEST_EDDIES = 10
N_COLUMNS = 5
# Under the MAX_RADIUS row, one row per box of this many degrees of longitude and latitude, centered on the eddy
MAP_DEGREES = (4, 8)
row_labels = [f'{MAX_RADIUS} R'] + [f'{box}° ×\n{box}°' for box in MAP_DEGREES]
plankton_chl = xr.open_mfdataset(sorted(resolve_bronze_dir(EXPERIMENT, 'plankton').glob('plankton_*.nc')), combine='by_coords')['CHL']
# Strength is the largest amplitude among the daily observations of the track
max_amplitude = cast(pd.Series, track_observations.groupby(identity_columns)['amplitude_cm'].max())
# `pigments` holds only the target eddies with an SDP retrieval, so every selected eddy has both products
strongest_eddies = pigments[identity_columns].drop_duplicates().join(max_amplitude, on=identity_columns).sort_values('amplitude_cm', ascending=False).groupby('polarity').head(N_STRONGEST_EDDIES)
# Layout in inches: maps 1.1 in wide and 1.15 in apart, a 0.34 in band for the dates above each line of maps, 0.14 in between rows, 0.32 in under each line for the eddy center position, 0.3 in for the eddy title, and 0.1 in below the last line; these exploratory figures ignore the AGU height limit
panel = 1.1
line_height = 0.34 + len(row_labels) * panel + (len(row_labels) - 1) * 0.14 + 0.32
# The Copernicus fields are daily, so each map averages the days of the PACE composite period, as the plankton gold table does; eddies share composites, so each regional mean is read once
region_chl: dict[pd.Timestamp, xr.DataArray] = {}
for polarity, color in polarity_colors.items():
    for track_id in strongest_eddies.loc[strongest_eddies['polarity'].eq(polarity), 'track_id']:
        eddy_track = cast(pd.DataFrame, track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(track_id)])
        snapshots = []
        # One map column per 8-day composite of the eddy in the plankton gold table
        for midpoint in plankton.loc[plankton['polarity'].eq(polarity) & plankton['track_id'].eq(track_id), 'date'].sort_values():
            # The contour the collocation used: nearest the composite midpoint, earlier day on a tie
            period_track = eddy_track.loc[eddy_track['date'].between(midpoint - pd.Timedelta(days=3), midpoint + pd.Timedelta(days=4))]
            contour = period_track.assign(midpoint_distance=(period_track['date'] - midpoint).abs()).sort_values(['midpoint_distance', 'date']).iloc[0]
            # The track center is smoothed along the track and can sit outside the contour of its own day, so every map centers on the area centroid of the contour
            contour_lon, contour_lat = np.asarray(contour['contour_lon']), np.asarray(contour['contour_lat'])
            cross = contour_lon * np.roll(contour_lat, -1) - np.roll(contour_lon, -1) * contour_lat
            center_lon = float(np.sum((contour_lon + np.roll(contour_lon, -1)) * cross) / (3 * cross.sum()))
            center_lat = float(np.sum((contour_lat + np.roll(contour_lat, -1)) * cross) / (3 * cross.sum()))
            # Degrees per speed radius: every map is drawn in degrees from the eddy center, and the first row reaches MAX_RADIUS speed radii, the radial unit of Figure 2
            lon_per_radius = float(contour['radius_km']) / (KM_PER_DEG_LAT * np.cos(np.radians(center_lat)))
            lat_per_radius = float(contour['radius_km']) / KM_PER_DEG_LAT
            contour_x = np.append(contour['contour_lon'], contour['contour_lon'][0]) - center_lon
            contour_y = np.append(contour['contour_lat'], contour['contour_lat'][0]) - center_lat
            daily_axis = streamline.loc[streamline['date'].eq(contour['date'])].sort_values('point_idx')
            if midpoint not in region_chl:
                region_chl[midpoint] = plankton_chl.sel(time=slice(midpoint - pd.Timedelta(days=3), midpoint + pd.Timedelta(days=4))).mean('time').compute()
            maps = []
            for half_lon, half_lat in [(MAX_RADIUS * lon_per_radius, MAX_RADIUS * lat_per_radius)] + [(box / 2, box / 2) for box in MAP_DEGREES]:
                # The box reaches 0.05 degrees past the map, just over one cell, so the edge cells fill the frame
                chl = region_chl[midpoint].sel(
                    longitude=slice(center_lon - half_lon - 0.05, center_lon + half_lon + 0.05),
                    latitude=slice(center_lat - half_lat - 0.05, center_lat + half_lat + 0.05),
                )
                maps.append((chl['longitude'].to_numpy() - center_lon, chl['latitude'].to_numpy() - center_lat, chl.to_numpy(), half_lon, half_lat))
            snapshots.append((
                midpoint, f'{-center_lon:.1f}°W\n{center_lat:.1f}°N', contour_x, contour_y,
                daily_axis['lon'].to_numpy() - center_lon, daily_axis['lat'].to_numpy() - center_lat, maps,
            ))

        # An eddy with more than N_COLUMNS composites wraps onto further lines
        figure_height = 0.3 + -(-len(snapshots) // N_COLUMNS) * line_height + 0.1
        fig = cast(Figure, plt.figure(figsize=(6.69, figure_height)))
        top = figure_height - 0.3
        fig.text(0.42 / 6.69, top / figure_height, f'{polarity.capitalize()} {track_id}, maximum amplitude {max_amplitude[polarity, track_id]:.0f} cm', color=color, fontweight='bold', va='bottom')
        # The date band less the 0.14 in gap that the loop adds above every row
        row_offset = 0.34 - 0.14
        for row, row_label in enumerate(row_labels):
            # Each row of an eddy has one scale for all its maps, from the 2nd to the 98th percentile of the mapped pixels
            low, high = np.nanpercentile(np.concatenate([maps[row][2].ravel() for *_, maps in snapshots]), [2, 98])
            # A wider map can reach shelf and slope water with many times the CHL around the eddy, so every scale is logarithmic; the ticks are the 1, 2, 5 values of each decade, or the decades alone when more than four of those fit
            decades = 10.0 ** np.arange(-3, 2)
            ticks = [tick for tick in np.outer(decades, [1, 2, 5]).ravel().tolist() if low <= tick <= high]
            if len(ticks) > 4:
                ticks = [tick for tick in ticks if tick in decades]
            if len(ticks) < 2:
                # A scale that holds fewer than two of those values is labeled at its two ends
                ticks = [float(low), float(high)]
            norm = LogNorm(low, high)  # pyright: ignore[reportCallIssue]
            row_offset += 0.14 + panel
            for position, (midpoint, center_label, contour_x, contour_y, axis_x, axis_y, maps) in enumerate(snapshots):
                line, column = divmod(position, N_COLUMNS)
                ax = fig.add_axes(((0.42 + 1.15 * column) / 6.69, (top - line * line_height - row_offset) / figure_height, panel / 6.69, panel / figure_height))
                x, y, values, half_lon, half_lat = maps[row]
                ax.pcolormesh(x, y, np.ma.masked_invalid(values), cmap='viridis', norm=norm, shading='nearest', linewidth=0, rasterized=True)
                # The Gulf Stream axis on the day of the contour, without the white edge so the eddy stands apart from it
                ax.plot(axis_x, axis_y, color='black', linewidth=0.8, solid_joinstyle='round')
                ax.plot(contour_x, contour_y, color='white', linewidth=1.8, solid_joinstyle='round')
                ax.plot(contour_x, contour_y, color='black', linewidth=0.8, solid_joinstyle='round')
                ax.set(xlim=(-half_lon, half_lon), ylim=(-half_lat, half_lat), xticks=[], yticks=[])
                for spine in ax.spines.values():
                    spine.set_visible(True)
                    spine.set_linewidth(0.5)
                if column == 0:
                    ax.text(-0.34 / panel, 0.5, row_label, transform=ax.transAxes, ha='left', va='center')
                if row == 0:
                    # The year shows at the start of each line and where it changes
                    year = f'{midpoint:%Y}\n' if column == 0 or midpoint.year != snapshots[position - 1][0].year else ''
                    ax.annotate(f'{year}{midpoint:%-d %b}', xy=(0.5, 1), xycoords='axes fraction', xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', linespacing=1.2)
                if row == len(row_labels) - 1:
                    ax.annotate(center_label, xy=(0.5, 0), xycoords='axes fraction', xytext=(0, -3), textcoords='offset points', ha='center', va='top', linespacing=1.2)
            bar = fig.colorbar(ScalarMappable(norm=norm, cmap='viridis'), cax=fig.add_axes(((0.44 + 1.15 * min(len(snapshots), N_COLUMNS)) / 6.69, (top - row_offset) / figure_height, 0.07 / 6.69, panel / figure_height)))
            bar.set_ticks(ticks, labels=[f'{tick:.2g}' for tick in ticks])
            bar.minorticks_off()
            bar.ax.tick_params(length=2, width=0.5)
            bar.outline.set_linewidth(0.5)  # pyright: ignore[reportAttributeAccessIssue]
            if row == 0:
                bar.ax.annotate('mg m$^{-3}$', xy=(0, 1), xycoords='axes fraction', xytext=(0, 3), textcoords='offset points', ha='left', va='bottom')
        plt.show()

**Figure 4.** The ten cyclones and ten anticyclones with the largest maximum amplitude among the target eddies with an SDP retrieval, strongest first, one figure per eddy with all of its eight-day composites in date order. Every map shows Copernicus chlorophyll-a, averaged over the eight days of the PACE composite, and is centered on the centroid of the speed contour. The three rows zoom out: the first map reaches two speed radii from the center in each direction, and the next two are boxes of 4° and 8° of longitude and latitude. The black line with a white edge is the speed contour, and the plain black line is the Gulf Stream axis of the contour's day where it crosses the map. Each row of an eddy has its own logarithmic color scale, set from the 2nd to the 98th percentile of the mapped pixels. The position under each column is the contour centroid.


In [ ]:
# The case study eddies picked on 2026-10-14 (docs/case_study_eddies.md), found by polarity and the eddy center on the day of
# maximum amplitude, so the picks survive a re-track that renumbers the tracks
CASE_STUDY_PICKS = (
    ('cyclone', '2025-12-24', -65.12, 36.53),
    ('anticyclone', '2024-06-29', -63.56, 39.92),
)
MAX_PICK_OFFSET_DEG = 0.5
destination_signs = {'cyclone': -1, 'anticyclone': 1}  # signed distance is positive north of the axis
daily_axes = {date: day.sort_values('point_idx') for date, day in streamline.groupby('date')}


def find_case_study(polarity: str, max_day: str, lon: float, lat: float) -> tuple[str, int, str, tuple[str, str]]:
    """Track id, the day it first reached the destination side, and the two composite midpoints to map: the last on the origin side and the last of the track."""
    day = track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['date'].eq(pd.Timestamp(max_day))]
    # Distance in degrees of latitude, with longitude scaled by cos(latitude)
    offset = np.hypot((day['center_lon'].to_numpy() - lon) * np.cos(np.radians(lat)), day['center_lat'].to_numpy() - lat)
    if offset.size == 0 or offset.min() > MAX_PICK_OFFSET_DEG:
        raise ValueError(f'no {polarity} within {MAX_PICK_OFFSET_DEG} degrees of the pick on {max_day}')
    track_id = int(day['track_id'].to_numpy()[offset.argmin()])
    track = cast(pd.DataFrame, track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(track_id)]).sort_values('date')
    signed_km = np.array([
        compute_signed_distance_km(daily_axes[date]['lon'].to_numpy(), daily_axes[date]['lat'].to_numpy(), x, y)[0]
        for date, x, y in zip(track['date'], track['center_lon'], track['center_lat'])
    ])
    on_destination = destination_signs[polarity] * signed_km > 0
    # First day on the destination side right after a day on the origin side
    entry_position = int(np.flatnonzero(on_destination[1:] & ~on_destination[:-1])[0]) + 1
    entry = cast(pd.Timestamp, track['date'].iloc[entry_position])
    composites = plankton.loc[plankton['polarity'].eq(polarity) & plankton['track_id'].eq(track_id), 'date'].sort_values()
    before_entry = composites.loc[composites.lt(entry)]
    origin_composite = before_entry.iloc[-1] if len(before_entry) else composites.iloc[0]
    return polarity, track_id, f'{entry:%Y-%m-%d}', (f'{origin_composite:%Y-%m-%d}', f'{composites.iloc[-1]:%Y-%m-%d}')


case_studies = tuple(find_case_study(*pick) for pick in CASE_STUDY_PICKS)
print(case_studies)
# Each case eddy's speed contour every CONTOUR_EVERY_DAYS days, colored by age as in Figure 1c, with the contours of the two mapped composites in black
case_tracks = [cast(pd.DataFrame, track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(track_id)]).sort_values('date') for polarity, track_id, *_ in case_studies]
# Both panels share one extent, so they have the same size and scale
case_lon = np.concatenate([np.concatenate(track['contour_lon'].to_list()) for track in case_tracks])
case_lat = np.concatenate([np.concatenate(track['contour_lat'].to_list()) for track in case_tracks])
case_extent = (np.floor(case_lon.min()) - 0.5, np.ceil(case_lon.max()) + 0.5, np.floor(case_lat.min()) - 0.5, np.ceil(case_lat.max()) + 0.5)
fig = cast(Figure, plt.figure(figsize=(6.69, 3.3)))
case_maps = fig.add_gridspec(1, 2, left=0.075, right=0.985, bottom=0.3, top=0.93, wspace=0.14)
for case_index, ((polarity, track_id, entry_date, snapshot_dates), track) in enumerate(zip(case_studies, case_tracks)):
    age_days = (track['date'] - track['date'].iloc[0]).dt.days
    age_norm = Normalize(0, int(age_days.iloc[-1]))
    ax = cast(GeoAxes, fig.add_subplot(case_maps[case_index], projection=ccrs.PlateCarree()))
    ax.set_extent(case_extent, crs=ccrs.PlateCarree())
    ax.plot(axis_lon, axis_lat, color='black', linewidth=1.0, linestyle='--', transform=ccrs.PlateCarree(), zorder=4)
    drawn = track.loc[age_days.mod(CONTOUR_EVERY_DAYS).eq(0) | track['date'].eq(track['date'].iloc[-1])]
    for contour_lon, contour_lat, age in zip(drawn['contour_lon'], drawn['contour_lat'], age_days.loc[drawn.index]):
        ax.plot(np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]), color=age_cmaps[polarity](age_norm(age)), linewidth=0.7, transform=ccrs.PlateCarree(), zorder=5)
    # The observation nearest each composite midpoint, earlier day on a tie, as in Figure 4
    for snapshot_date in snapshot_dates:
        mapped = track.iloc[int((track['date'] - pd.Timestamp(snapshot_date)).abs().argmin())]
        ax.plot(np.append(mapped['contour_lon'], mapped['contour_lon'][0]), np.append(mapped['contour_lat'], mapped['contour_lat'][0]), color='black', linewidth=1.3, transform=ccrs.PlateCarree(), zorder=6)
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=2)
    ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
    gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', xlocs=range(-80, -55, 2), ylocs=range(30, 45, 2), zorder=3)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.left_labels = case_index == 0
    gridlines.xlabel_style = {'size': 8}
    gridlines.ylabel_style = {'size': 8}
    age_bar = fig.colorbar(ScalarMappable(norm=age_norm, cmap=age_cmaps[polarity]), cax=ax.inset_axes((0, -0.24, 1, 0.05)), orientation='horizontal')
    age_bar.set_label('Age (days)')
    age_bar.ax.xaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 5, 10]))
    age_bar.ax.tick_params(length=2)
    age_bar.outline.set_linewidth(0.6)  # pyright: ignore[reportAttributeAccessIssue]
    ax.set_title(f'$\\bf{{({chr(97 + case_index)})}}$ {polarity.capitalize()}', loc='left', color=polarity_colors[polarity])
fig.legend(handles=[
    Line2D([], [], color='black', linewidth=1.3, label='Contour of a mapped composite'),
    Line2D([], [], color='black', linewidth=1.0, linestyle='--', label='Mean Gulf Stream axis'),
], loc='lower center', bbox_to_anchor=(0.5, 0), ncol=2, columnspacing=2.5, handlelength=2, borderaxespad=0.2)
plt.show()

Filter to only targets + stronger eddies (speed, amplitude, etc) to visualize snapshots for each eddy

https://www.youtube.com/watch?v=AiRPInRGgfM

Q1:
Clusters make taxonomical sense in ecosystem context
See if Rrs is within range of training spectra, not necessarily median; see what percentage of spectra north/south are within training range

Q2:
Phytoplankton may be less indicative of change in water mass
Evolution of community may be different from evolution of water mass

Q3:
Diff combinations of variables can create same can give same Rrs pattern
Can try; classify per-pixel which cluster it belongs to in terms of north/south water Rrs

Q4:
Write short description of figures and send it to someone at NASA??
Difference in coverage over the year is not fixable; just happens
What might need to be fixed is that calibration for remote sensing data is not properly accounting for season; 
Affects global/regional analysis over long periods of time

Getting spectral shape in seasonal variability; shows phytoplankton absorption trend varying over season but also over both regions



![](images/2026-10-26-10-32-22.png)